In [1]:
import json
import re
from pathlib import Path
from copy import deepcopy


INPUT_PATH = "/home/vcnt/Repositories/Tesis/datasets_metadata/histai_colorectal_b2_metadata.json"
OUTPUT_PATH = "/home/vcnt/Repositories/Tesis/datasets_metadata/histai_colorectal_b2_metadata_regroup.json"

def normalize_text(s):
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = s.replace("c-r", "cancer")
    s = s.replace("cr ", "cancer ")
    s = s.replace("crc", "colorectal cancer")
    s = re.sub(r'[^a-z0-9\s\?\-]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def detect_site(s):
    site_rules = [
        ("rectosigmoid_junction", [r"rectosigmoid junction", r"rectosigmoid"]),
        ("rectum", [r"\brectal\b", r"\brectum\b", r"ampulla", r"ampullary"]),
        ("sigmoid_colon", [r"sigmoid colon", r"\bsigmoid\b"]),
        ("descending_colon", [r"descending colon"]),
        ("transverse_colon", [r"transverse colon"]),
        ("ascending_colon", [r"ascending colon", r"right colon"]),
        ("cecum", [r"\bcecal\b", r"\bcecum\b", r"cecal dome"]),
        ("anal_canal", [r"anal canal", r"\banal\b", r"anal sphincter"]),
        ("colon_unspecified", [r"\bcolon\b", r"\bcolorectal\b", r"\bcolonic\b", r"\bintest"]),
    ]
    for label, patterns in site_rules:
        if any(re.search(p, s) for p in patterns):
            return label
    return "unspecified"

def has_uncertainty(s):
    return any(x in s for x in ["?", "suspicious", "suspicion", "suspected", "possible"])

def classify_diagnosis(raw_dx):
    s = normalize_text(raw_dx)
    site = detect_site(s)

    if s in {"", "-", "examination", "observation"}:
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "missing_or_uninformative",
            "taxonomy_subgroup": "missing_or_unspecified",
            "taxonomy_site": site,
            "taxonomy_status": "review",
            "taxonomy_rule": "uninformative"
        }

    if any(x in s for x in ["stomach", "gastric", "antrum of the stomach", "prostate"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "non_colorectal_or_noise",
            "taxonomy_subgroup": "other_organ_or_noise",
            "taxonomy_site": site,
            "taxonomy_status": "review",
            "taxonomy_rule": "non_colorectal"
        }

    if any(x in s for x in ["ulcerative colitis", "colitis", "proctitis", "ileitis"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "inflammatory_or_non_neoplastic",
            "taxonomy_subgroup": "colitis_or_proctitis",
            "taxonomy_site": site,
            "taxonomy_status": "final",
            "taxonomy_rule": "inflammatory"
        }

    if "adenocarcinoma" in s:
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "adenocarcinoma",
            "taxonomy_site": site,
            "taxonomy_status": "final",
            "taxonomy_rule": "adenocarcinoma"
        }

    if any(x in s for x in ["cancer", "malignant neoplasm", "carcinoma"]) and not has_uncertainty(s):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "colorectal_carcinoma_nos",
            "taxonomy_site": site,
            "taxonomy_status": "final",
            "taxonomy_rule": "crc_nos"
        }

    if has_uncertainty(s) and any(x in s for x in ["cancer", "tumor", "neoplasm", "cr"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "suspicious_or_uncertain",
            "taxonomy_subgroup": "suspicious_for_malignancy",
            "taxonomy_site": site,
            "taxonomy_status": "review",
            "taxonomy_rule": "suspicious_malignancy"
        }

    if any(x in s for x in ["adenoma", "benign neoplasm"]) and "adenocarcinoma" not in s:
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "benign_precursor",
            "taxonomy_subgroup": "adenoma",
            "taxonomy_site": site,
            "taxonomy_status": "final",
            "taxonomy_rule": "adenoma"
        }

    if any(x in s for x in ["polyp", "polyps", "polypoid"]) and "cancer" not in s:
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "benign_precursor",
            "taxonomy_subgroup": "polyp_nos",
            "taxonomy_site": site,
            "taxonomy_status": "review" if has_uncertainty(s) else "final",
            "taxonomy_rule": "polyp"
        }

    if any(x in s for x in ["epithelial formation", "epithelial formations", "formation", "neoplasm", "tumor", "mass"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "nonspecific_neoplastic",
            "taxonomy_subgroup": "epithelial_formation_or_neoplasm",
            "taxonomy_site": site,
            "taxonomy_status": "review",
            "taxonomy_rule": "nonspecific_neoplastic"
        }

    return {
        "diagnosis_normalized": s,
        "taxonomy_group": "non_colorectal_or_noise",
        "taxonomy_subgroup": "unmapped_review",
        "taxonomy_site": site,
        "taxonomy_status": "review",
        "taxonomy_rule": "fallback"
    }

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

out = []
for row in data:
    row2 = deepcopy(row)
    mapped = classify_diagnosis(row.get("diagnosis", ""))
    row2.update(mapped)
    out.append(row2)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print(f"Saved {len(out)} records to {OUTPUT_PATH}")

Saved 57 records to /home/vcnt/Repositories/Tesis/datasets_metadata/histai_colorectal_b2_metadata_regroup.json
